# Public release note

This is a sanitized public copy of `omniASR_20_K1_KenLM_inventory(1).ipynb`. Cell outputs, execution counters,
Colab user metadata, widget state, embedded display payloads, private storage
paths, and internal run identifiers were removed before publication. The
experiment source is preserved, but public placeholders must be configured from
your own generated contracts before rerunning dependent stages. See the repository
README and `docs/REPRODUCIBILITY.md` for prerequisites, data access, execution
order, expected artifacts, and the English explanation of this experiment.

Never paste credentials into a notebook. Use Colab Secrets or environment
variables (`HF_TOKEN`, `KAGGLE_USERNAME`, and `KAGGLE_KEY`).


# 20.K1 — Inventaire KenLM V6 (text-only)

Cette étape inventorie et fige les sources textuelles sans télécharger les audios ni les corpus complets.

Avant exécution : révoquer les anciens jetons joints à la conversation, créer de nouveaux jetons, puis ajouter dans les Secrets Colab `HF_TOKEN`, `KAGGLE_USERNAME` et `KAGGLE_KEY`. Accepter aussi les conditions des jeux de données Hugging Face gated.

Exécuter ensuite la cellule ci-dessous. Le statut attendu est `PASS_READY_FOR_20_K2`.


In [ ]:
# 20.K1 — Inventaire KenLM V6, textes et métadonnées uniquement
#
# Objectif
# --------
# Figer les sources textuelles, revisions, licences, splits et chemins de
# manifests utilisables pour KenLM, sans telecharger d'audio ni de dataset
# complet. Cette cellule ne construit pas encore de modele de langue.
#
# Secrets acceptes (sans jamais etre affiches ni sauvegardes)
# ----------------------------------------------------------
# Colab Secrets : HF_TOKEN, KAGGLE_USERNAME, KAGGLE_KEY (recommande), ou
# KAGGLE_API_TOKEN. Aucun fichier de credentials n'est lu par cette cellule.
#
# IMPORTANT : un token Hugging Face ne remplace pas l'acceptation manuelle des
# conditions d'un dataset gated. Accepter d'abord les conditions sur sa page.

from __future__ import annotations

import csv
import hashlib
import importlib.util
import importlib.metadata
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
import time
from datetime import datetime, timezone
from pathlib import Path
from collections.abc import Mapping
from typing import Any
from urllib.parse import urlsplit, urlunsplit


# -----------------------------------------------------------------------------
# 0. Dependances legeres seulement ; aucun paquet audio n'est installe/importé
# -----------------------------------------------------------------------------

REQUIRED_MODULES = {
    "huggingface_hub": "huggingface_hub>=0.30.0",
    "requests": "requests>=2.31.0",
    "pandas": "pandas>=2.0.0",
    "pyarrow": "pyarrow>=14.0.0",
    "kaggle": "kaggle==2.2.3",
}

missing_packages = []
for module_name, requirement in REQUIRED_MODULES.items():
    if importlib.util.find_spec(module_name) is None:
        missing_packages.append(requirement)

try:
    if importlib.metadata.version("kaggle") != "2.2.3":
        missing_packages.append("kaggle==2.2.3")
except importlib.metadata.PackageNotFoundError:
    pass

MINIMUM_VERSIONS = {
    "huggingface_hub": ((0, 30, 0), "huggingface_hub>=0.30.0"),
    "requests": ((2, 31, 0), "requests>=2.31.0"),
    "pandas": ((2, 0, 0), "pandas>=2.0.0"),
    "pyarrow": ((14, 0, 0), "pyarrow>=14.0.0"),
}
for distribution, (minimum, requirement) in MINIMUM_VERSIONS.items():
    try:
        installed = importlib.metadata.version(distribution)
        numeric = tuple(
            int(value)
            for value in (re.findall(r"\d+", installed)[:3] + ["0", "0", "0"])
        )[:3]
        if numeric < minimum:
            missing_packages.append(requirement)
    except importlib.metadata.PackageNotFoundError:
        pass

missing_packages = list(dict.fromkeys(missing_packages))

if missing_packages:
    print("Installation des dependances metadata-only :", missing_packages)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *missing_packages],
        check=True,
    )

import pandas as pd
import pyarrow.parquet as pq
import requests
from huggingface_hub import HfApi, HfFileSystem


# -----------------------------------------------------------------------------
# 1. Drive, chemins et constantes
# -----------------------------------------------------------------------------

try:
    from google.colab import drive

    if not Path("/content/persistent_storage").exists():
        drive.mount("/content/drive")
except ImportError:
    pass

FT_ROOT = Path(
    os.environ.get(
        "AFRIVOICES_FT_ROOT",
        "/content/afrivoices_project/"
        "finetune_runs/experiment_run",
    )
)
PROJECT_ROOT = FT_ROOT.parent.parent
REPORT_ROOT = FT_ROOT / "reports/kenlm_v6_20_K1"
COMPETITION = "afri-voices-east-africa-asr-hackathon"

assert Path("/content/persistent_storage").exists(), "Monter Google Drive avant 20.K1."
assert FT_ROOT.exists(), f"Racine du projet introuvable : {FT_ROOT}"
REPORT_ROOT.mkdir(parents=True, exist_ok=True)

LANGUAGES = ("sw", "kik", "kln", "luo", "som", "mas")
LANGUAGE_ALIASES = {
    "sw": ("sw", "swa", "swh", "swahili", "kiswahili"),
    "kik": ("kik", "ki", "kikuyu", "gikuyu"),
    "kln": ("kln", "sgc", "niq", "kalenjin", "kipsigis", "nandi"),
    "luo": ("luo", "dholuo"),
    "som": ("som", "so", "somali", "soomali", "af-soomaali"),
    "mas": ("mas", "maasai", "maa", "saq"),
}

APPROVED_LICENSES = {
    "apache-2.0",
    "cc-by-4.0",
    "cc0-1.0",
    "cc0",
    "mit",
}
REVIEW_LICENSES = {
    "cc-by-sa-4.0",
    "cc-by-sa-3.0",
    "gfdl",
    "other",
}

TEXT_EXTENSIONS = {
    ".csv", ".tsv", ".json", ".jsonl", ".ndjson", ".parquet", ".arrow",
    ".txt", ".text", ".xml", ".tmx",
}
AUDIO_EXTENSIONS = {
    ".wav", ".flac", ".mp3", ".ogg", ".opus", ".m4a", ".aac", ".wma",
}
ARCHIVE_EXTENSIONS = {
    ".zip", ".tar", ".gz", ".tgz", ".bz2", ".xz", ".7z", ".zst",
}
TEXT_NAME_HINTS = (
    "transcript", "sentence", "metadata", "manifest", "label", "prompt",
    "text", "translation", "utterance", "annotation",
)


# -----------------------------------------------------------------------------
# 2. Registre conservateur des sources
# -----------------------------------------------------------------------------

# ingest_policy :
# - extract       : candidat direct pour 20.K2
# - mirror        : alternative au depot principal ; ne jamais compter deux fois
# - catalog_only  : inventaire/licence seulement, pas d'ingestion automatique

SOURCE_REGISTRY: list[dict[str, Any]] = [
    {
        "source_id": "anv_ke_catalog",
        "repo_id": "MCAA1-MSU/anv_data_ke",
        "languages": ["kik", "kln", "luo", "som", "mas"],
        "family": "anv_ke",
        "priority": 1,
        "core": True,
        "gated_expected": True,
        "expected_license": "cc-by-4.0",
        "kind": "speech_transcripts",
        "ingest_policy": "extract",
        "expected_text_fields": ["actualSentence", "transcript"],
        "notes": "Depot ANV Kenya principal ; train uniquement pour KenLM.",
    },
    {
        "source_id": "anv_ke_kikuyu_mirror",
        "repo_id": "Anv-ke/Kikuyu",
        "languages": ["kik"],
        "family": "anv_ke",
        "priority": 1,
        "core": False,
        "gated_expected": True,
        "expected_license": "cc-by-4.0",
        "kind": "speech_transcripts",
        "ingest_policy": "mirror",
        "expected_text_fields": ["actualSentence", "transcript"],
        "notes": "Miroir par langue ; utiliser seulement si preferable au depot principal.",
    },
    {
        "source_id": "anv_ke_kalenjin_mirror",
        "repo_id": "Anv-ke/Kalenjin",
        "languages": ["kln"],
        "family": "anv_ke",
        "priority": 1,
        "core": False,
        "gated_expected": True,
        "expected_license": "cc-by-4.0",
        "kind": "speech_transcripts",
        "ingest_policy": "mirror",
        "expected_text_fields": ["actualSentence", "transcript"],
        "notes": "Miroir par langue ; ne pas doubler avec anv_ke_catalog.",
    },
    {
        "source_id": "anv_ke_dholuo_mirror",
        "repo_id": "Anv-ke/Dholuo",
        "languages": ["luo"],
        "family": "anv_ke",
        "priority": 1,
        "core": False,
        "gated_expected": True,
        "expected_license": "cc-by-4.0",
        "kind": "speech_transcripts",
        "ingest_policy": "mirror",
        "expected_text_fields": ["actualSentence", "transcript"],
        "notes": "Miroir par langue ; ne pas doubler avec anv_ke_catalog.",
    },
    {
        "source_id": "anv_ke_somali_mirror",
        "repo_id": "Anv-ke/Somali",
        "languages": ["som"],
        "family": "anv_ke",
        "priority": 1,
        "core": False,
        "gated_expected": True,
        "expected_license": "cc-by-4.0",
        "kind": "speech_transcripts",
        "ingest_policy": "mirror",
        "expected_text_fields": ["actualSentence", "transcript"],
        "notes": "Miroir par langue ; ne pas doubler avec anv_ke_catalog.",
    },
    {
        "source_id": "anv_ke_maasai_mirror",
        "repo_id": "Anv-ke/Maasai",
        "languages": ["mas"],
        "family": "anv_ke",
        "priority": 1,
        "core": False,
        "gated_expected": True,
        "expected_license": "cc-by-4.0",
        "kind": "speech_transcripts",
        "ingest_policy": "mirror",
        "expected_text_fields": ["actualSentence", "transcript"],
        "notes": "Miroir par langue ; ne pas doubler avec anv_ke_catalog.",
    },
    {
        "source_id": "afrivoice_swahili",
        "repo_id": "DigitalUmuganda/Afrivoice_Swahili",
        "languages": ["sw"],
        "family": "afrivoice_swahili",
        "priority": 1,
        "core": True,
        "gated_expected": True,
        "expected_license": "cc-by-4.0",
        "kind": "speech_transcripts",
        "ingest_policy": "extract",
        "expected_text_fields": ["text", "sentence", "transcription", "transcript"],
        "notes": "Cinq domaines Swahili ; train uniquement.",
    },
    {
        "source_id": "digigreen_kikuyu",
        "repo_id": "DigiGreen/KikuyuASR_trainingdataset",
        "languages": ["kik"],
        "family": "digigreen_kikuyu",
        "priority": 2,
        "core": False,
        "gated_expected": False,
        "expected_license": "apache-2.0",
        "kind": "speech_transcripts",
        "ingest_policy": "extract",
        "expected_text_fields": ["text", "sentence", "transcription", "transcript"],
        "notes": "Phrases agriculture/read ; verifier le format CSV avant 20.K2.",
    },
    {
        "source_id": "thinkkenya_parallel",
        "repo_id": "thinkKenya/kenyan-low-resource-language-data",
        "languages": ["sw", "kln", "luo"],
        "family": "thinkkenya_parallel",
        "priority": 2,
        "core": False,
        "gated_expected": True,
        "expected_license": "cc-by-4.0",
        "kind": "parallel_text",
        "ingest_policy": "extract",
        "expected_configs_by_language": {
            "sw": ["kln_swa", "luo_swa"],
            "kln": ["kln_swa"],
            "luo": ["luo_swa"],
        },
        "expected_text_fields": ["translation"],
        "notes": "Utiliser seulement train ; extraire le cote de la langue cible.",
    },
    {
        "source_id": "kencorpus_text",
        "repo_id": "Kencorpus/KenCorpus_text",
        "languages": ["sw", "luo"],
        "family": "kencorpus_text",
        "priority": 2,
        "core": False,
        "gated_expected": False,
        "expected_license": "cc-by-4.0",
        "kind": "community_text",
        "ingest_policy": "extract",
        "expected_text_fields": ["text"],
        "notes": "Community/news ; filtrer les artefacts de conversion documentaire.",
    },
    {
        "source_id": "fleurs",
        "repo_id": "google/fleurs",
        "languages": ["sw", "luo", "som"],
        "family": "fleurs",
        "priority": 2,
        "core": False,
        "gated_expected": False,
        "expected_license": "cc-by-4.0",
        "kind": "speech_transcripts",
        "ingest_policy": "extract",
        "expected_configs": ["sw_ke", "luo_ke", "so_so"],
        "expected_configs_by_language": {
            "sw": ["sw_ke"],
            "luo": ["luo_ke"],
            "som": ["so_so"],
        },
        "expected_text_fields": ["transcription", "raw_transcription"],
        "notes": "Train uniquement ; petit corpus propre.",
    },
    {
        "source_id": "somali_asr_public",
        "repo_id": "skydheere/soomali-asr-dataset",
        "languages": ["som"],
        "family": "somali_asr_public",
        "priority": 2,
        "core": False,
        "gated_expected": False,
        "expected_license": "cc-by-4.0",
        "kind": "speech_transcripts",
        "ingest_policy": "extract",
        "expected_text_fields": ["text", "sentence", "transcription", "transcript"],
        "notes": "Auditer provenance et doublons avant ingestion.",
    },
    {
        "source_id": "omnilingual_asr_corpus",
        "repo_id": "facebook/omnilingual-asr-corpus",
        "languages": ["sw", "kik", "kln", "luo", "som", "mas"],
        "family": "omnilingual_asr_corpus",
        "priority": 2,
        "core": False,
        "gated_expected": False,
        "expected_license": "cc-by-4.0",
        "kind": "speech_transcripts",
        "ingest_policy": "extract",
        "expected_text_fields": ["raw_text"],
        "notes": "20.K2 devra confirmer les codes de langue reellement presents.",
    },
    {
        "source_id": "maasai_translation_public",
        "repo_id": "NorthernTribe-Research/maasai-translation-corpus",
        "languages": ["mas"],
        "family": "maasai_translation_public",
        "priority": 3,
        "core": False,
        "gated_expected": True,
        "expected_license": "cc-by-4.0",
        "kind": "parallel_text",
        "ingest_policy": "extract",
        "expected_text_fields": ["translation", "maasai", "mas"],
        "notes": "Optionnel ; verifier variete Maa et langue-ID.",
    },
    {
        "source_id": "waxal_catalog_review",
        "repo_id": "google/WaxalNLP",
        "languages": ["sw", "kik", "luo"],
        "family": "waxal",
        "priority": 4,
        "core": False,
        "gated_expected": False,
        "expected_license": "cc-by-sa-4.0",
        "kind": "speech_and_tts_transcripts",
        "ingest_policy": "catalog_only",
        "expected_text_fields": ["transcription", "text"],
        "notes": "REVIEW licence Share-Alike ; ne pas ingerer automatiquement.",
    },
]

# Sources textuelles supplementaires retenues par l'audit conservateur.
SOURCE_REGISTRY.extend(
    [
        {
            "source_id": "kenspeech_text",
            "repo_id": "Kencorpus/KenSpeech",
            "languages": ["sw"],
            "family": "kencorpus_kenspeech",
            "priority": 2,
            "core": False,
            "gated_expected": False,
            "expected_license": "cc-by-4.0",
            "kind": "speech_transcripts",
            "ingest_policy": "extract",
            "expected_splits": ["train"],
            "expected_text_fields": ["transcript"],
            "expected_text_files": ["transcripts_only.csv"],
            "lexicon_files": ["lexicon.csv"],
            "notes": "20.K2 ne lira que les colonnes texte, jamais l'audio Parquet.",
        },
        {
            "source_id": "kenpos_dholuo",
            "repo_id": "Kencorpus/KenPOS",
            "languages": ["luo"],
            "family": "kencorpus_kenpos",
            "priority": 3,
            "core": False,
            "gated_expected": False,
            "expected_license": "cc-by-4.0",
            "kind": "tokenized_text",
            "ingest_policy": "extract",
            "expected_configs": ["dho"],
            "expected_configs_by_language": {"luo": ["dho"]},
            "expected_splits": ["train"],
            "expected_text_fields": ["token", "filename", "sentence_id", "position"],
            "notes": "Reconstruire chaque phrase par (filename,sentence_id), triee par position.",
        },
        {
            "source_id": "google_smol",
            "repo_id": "google/smol",
            "languages": ["sw", "kik", "luo", "som"],
            "family": "google_smol",
            "priority": 3,
            "core": False,
            "gated_expected": False,
            "expected_license": "cc-by-4.0",
            "kind": "parallel_text",
            "ingest_policy": "extract",
            "expected_configs": ["en_sw", "en_ki", "en_luo", "en_so"],
            "expected_configs_by_language": {
                "sw": ["en_sw"],
                "kik": ["en_ki"],
                "luo": ["en_luo"],
                "som": ["en_so"],
            },
            "expected_splits": ["train"],
            "expected_text_fields": ["trg", "trgs"],
            "notes": "Faible poids LM ; GATITOS sert uniquement au vocabulaire unigramme.",
        },
        {
            "source_id": "xnli_extension_review",
            "repo_id": "tonative/xnli-extension",
            "languages": ["kik", "luo"],
            "family": "xnli_extension",
            "priority": 4,
            "core": False,
            "gated_expected": False,
            "expected_license": "cc-by-4.0",
            "kind": "translated_text",
            "ingest_policy": "catalog_only",
            "provenance_status": "review",
            "expected_splits": ["train"],
            "notes": "Release communautaire recente : schema, LID et provenance a valider.",
        },
        {
            "source_id": "swahili_corpus_repack_review",
            "repo_id": "ngusadeep/Swahili-Corpus-Dataset",
            "languages": ["sw"],
            "family": "swahili_corpus_repack",
            "priority": 4,
            "core": False,
            "gated_expected": False,
            "expected_license": "apache-2.0",
            "kind": "web_and_ocr_text",
            "ingest_policy": "catalog_only",
            "provenance_status": "review",
            "expected_splits": ["train"],
            "expected_text_fields": ["text"],
            "notes": "Grand repack bruité : n'activer qu'apres provenance, LID et dedup pousses.",
        },
    ]
)

# La licence seule ne suffit pas : seules les sources a provenance approuvee
# pourront passer automatiquement en 20.K2.
for _source in SOURCE_REGISTRY:
    _source.setdefault("provenance_status", "approved")
    _source.setdefault("expected_splits", ["train"])

for _source in SOURCE_REGISTRY:
    if _source["source_id"] in {
        "digigreen_kikuyu",
        "somali_asr_public",
        "omnilingual_asr_corpus",
        "maasai_translation_public",
        "waxal_catalog_review",
    }:
        _source["ingest_policy"] = "catalog_only"
        _source["provenance_status"] = "review"

MANUAL_EXTERNAL_SOURCES = [
    {
        "source_id": "common_voice_swahili",
        "languages": ["sw"],
        "provider": "Mozilla Data Collective",
        "url": "https://mozilladatacollective.com/organization/cmfh0j9o10006ns07jq45h7xk",
        "expected_license": "cc0-1.0",
        "status": "MANUAL_ACCESS_REQUIRED",
        "notes": "Exporter seulement les TSV de metadonnees/transcriptions.",
    },
    {
        "source_id": "common_voice_kalenjin",
        "languages": ["kln"],
        "provider": "Mozilla Data Collective",
        "url": "https://mozilladatacollective.com/datasets/cmqinqp7r00x2nq071z5e06oa",
        "expected_license": "cc0-1.0",
        "status": "MANUAL_ACCESS_REQUIRED",
        "notes": "Exporter seulement les TSV de metadonnees/transcriptions.",
    },
    {
        "source_id": "common_voice_dholuo",
        "languages": ["luo"],
        "provider": "Mozilla Data Collective",
        "url": "https://mozilladatacollective.com/datasets/cmqinqej600xcnr07kx4oxaxf",
        "expected_license": "cc0-1.0",
        "status": "MANUAL_ACCESS_REQUIRED",
        "notes": "Exporter seulement les TSV de metadonnees/transcriptions.",
    },
]


# -----------------------------------------------------------------------------
# 3. Utilitaires : serialisation, hash, secrets et redaction
# -----------------------------------------------------------------------------

def canonical_json(value: Any) -> str:
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    )


def sha256_bytes(value: bytes) -> str:
    return hashlib.sha256(value).hexdigest()


def sha256_file(path: Path, block_size: int = 8 << 20) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


def atomic_json(value: Any, path: Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")
    with open(temporary, "w", encoding="utf-8") as handle:
        json.dump(value, handle, ensure_ascii=False, indent=2, default=str)
    os.replace(temporary, path)
    print("json ->", path)


def atomic_csv(records: list[dict[str, Any]], path: Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    frame = pd.DataFrame(records)
    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")
    frame.to_csv(temporary, index=False, encoding="utf-8")
    os.replace(temporary, path)


def string_list(value: Any) -> list[str]:
    if value is None:
        return []
    values = value if isinstance(value, (list, tuple, set)) else [value]
    return [str(item).strip() for item in values if str(item).strip()]


def normalize_license(value: Any) -> list[str]:
    if value is None:
        return []
    values = value if isinstance(value, (list, tuple, set)) else [value]
    output = []
    for item in values:
        text = str(item).strip().lower().replace("_", "-")
        text = re.sub(r"\s+", "-", text)
        aliases = {
            "cc-by-4": "cc-by-4.0",
            "cc-by-4-0": "cc-by-4.0",
            "cc0-1": "cc0-1.0",
            "apache-2": "apache-2.0",
            "cc-by-sa-4": "cc-by-sa-4.0",
        }
        text = aliases.get(text, text)
        if text and text not in output:
            output.append(text)
    return output


def license_decision(reported: list[str], expected: str | None) -> str:
    values = set(normalize_license(reported))
    # La licence attendue sert seulement a detecter une divergence. Une carte
    # sans licence declaree reste UNKNOWN et ne peut jamais etre auto-approuvee.
    if len(values) > 1:
        return "REVIEW"
    if values and values.issubset(APPROVED_LICENSES):
        return "APPROVED"
    if values & REVIEW_LICENSES:
        return "REVIEW"
    return "UNKNOWN"


def get_colab_secret(name: str) -> str | None:
    try:
        from google.colab import userdata

        value = userdata.get(name)
        return str(value).strip() if value else None
    except Exception:
        return None


HF_TOKEN = (
    os.environ.get("HF_TOKEN")
    or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    or get_colab_secret("HF_TOKEN")
)
KAGGLE_USERNAME = (
    os.environ.get("KAGGLE_USERNAME")
    or get_colab_secret("KAGGLE_USERNAME")
)
KAGGLE_KEY = (
    os.environ.get("KAGGLE_KEY")
    or get_colab_secret("KAGGLE_KEY")
)
KAGGLE_API_TOKEN = (
    os.environ.get("KAGGLE_API_TOKEN")
    or get_colab_secret("KAGGLE_API_TOKEN")
)

SECRET_VALUES = [
    str(value)
    for value in (HF_TOKEN, KAGGLE_USERNAME, KAGGLE_KEY, KAGGLE_API_TOKEN)
    if value
]


def redact(value: Any) -> str:
    text = str(value)
    for secret in SECRET_VALUES:
        if secret:
            text = text.replace(secret, "<REDACTED>")
    text = re.sub(r"(?i)(authorization|token|key)=?[^\s,;]+", r"\1=<REDACTED>", text)
    return text[:1000]


KAGGLE_CREDS_PRESENT = bool(
    KAGGLE_API_TOKEN or (KAGGLE_USERNAME and KAGGLE_KEY)
)

print("Secrets detectes : HF=", bool(HF_TOKEN), "| Kaggle=", KAGGLE_CREDS_PRESENT)
print("Aucune valeur de secret ne sera affichee ou ecrite sur Drive.")


# -----------------------------------------------------------------------------
# 4. Inventaire des artefacts locaux deja figes sur Drive
# -----------------------------------------------------------------------------

LOCAL_CANDIDATES = {
    "selected_records_v4": FT_ROOT / "reports/balanced_v4/selected_records_v4.parquet",
    "selected_dev_v4": FT_ROOT / "reports/balanced_v4/selected_dev_v4.parquet",
    "balanced_dataset_complete": PROJECT_ROOT / "dataset_caches/afrivoices_balanced_v4_dataset/_COMPLETE.json",
    "v5_preflight_latest": FT_ROOT / "reports/v5_preflight_20_1/LATEST.json",
    "kenlm_historical_036529": FT_ROOT / "reports/kenlm_tuning_by_domain_v3_pre_step1250.json",
    "kenlm_score_036880": FT_ROOT / "reports/kenlm_tuning_by_domain_v3_score_036880.json",
    "kenlm_lora_036878": FT_ROOT / "reports/kenlm_tuning_by_domain_v3_lora_step1250_candidate.json",
    "leaderboard_036880": FT_ROOT / "reports/leaderboard_result_v4_036880.json",
    "kenlm_models_v1": FT_ROOT / "kenlm_models",
    "kenlm_models_v4": FT_ROOT / "kenlm_models_v4",
    "kenlm_models_v5": FT_ROOT / "kenlm_models_v5",
    "kenlm_models_web": FT_ROOT / "kenlm_models_web",
}


def inspect_json_gate(path: Path) -> dict[str, Any]:
    result = {
        "valid_json": False,
        "statuses": [],
        "top_level_status": None,
        "has_pass_status": False,
        "has_fail_status": False,
        "integrity_sha_count": 0,
        "languages_found": [],
        "error": None,
    }
    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
        if not isinstance(payload, (dict, list)):
            raise ValueError("JSON racine non structure")
        result["valid_json"] = True
        if isinstance(payload, Mapping) and payload.get("status") is not None:
            result["top_level_status"] = str(payload.get("status"))

        statuses: list[str] = []
        sha_values: list[str] = []
        language_values: list[str] = []

        def visit(value: Any, key: str = "") -> None:
            if isinstance(value, Mapping):
                for child_key, child_value in value.items():
                    child_key_text = str(child_key)
                    if "status" in child_key_text.lower() and isinstance(
                        child_value, (str, int, float, bool)
                    ):
                        statuses.append(str(child_value))
                    if child_key_text.lower() in {"language", "languages"}:
                        language_values.extend(string_list(child_value))
                    visit(child_value, child_key_text)
            elif isinstance(value, list):
                for child in value:
                    visit(child, key)
            elif isinstance(value, str) and re.fullmatch(
                r"[0-9a-fA-F]{64}", value.strip()
            ):
                sha_values.append(value.strip().lower())

        visit(payload)
        normalized_statuses = sorted(set(statuses))
        result["statuses"] = normalized_statuses
        top_status = str(result["top_level_status"] or "").upper()
        result["has_pass_status"] = top_status.startswith("PASS")
        result["has_fail_status"] = any(
            any(
                word in str(status).upper()
                for word in ("FAIL", "ERROR", "BLOCK", "REVIEW")
            )
            for status in normalized_statuses
        ) or any(word in top_status for word in ("FAIL", "ERROR", "BLOCK", "REVIEW"))
        result["integrity_sha_count"] = len(set(sha_values))
        canonical_languages = set()
        for value in language_values:
            normalized = value.strip().lower()
            for language, aliases in LANGUAGE_ALIASES.items():
                if normalized == language or normalized in aliases:
                    canonical_languages.add(language)
        result["languages_found"] = sorted(canonical_languages)
    except Exception as exc:
        result["error"] = redact(f"{type(exc).__name__}: {exc}")
    return result


def inspect_parquet_gate(path: Path) -> dict[str, Any]:
    result = {
        "valid_parquet": False,
        "rows": 0,
        "columns": [],
        "languages_found": [],
        "splits_found": [],
        "all_languages_present": False,
        "train_present": False,
        "schema_ok": False,
        "error": None,
    }
    try:
        parquet_file = pq.ParquetFile(path)
        columns = list(parquet_file.schema_arrow.names)
        rows = int(parquet_file.metadata.num_rows)
        normalized = {column.lower() for column in columns}
        original_by_lower = {column.lower(): column for column in columns}
        languages_found: list[str] = []
        splits_found: list[str] = []
        if "language" in original_by_lower and "split" in original_by_lower:
            gate_table = pq.read_table(
                path,
                columns=[
                    original_by_lower["language"],
                    original_by_lower["split"],
                ],
            )
            languages_found = sorted(
                {
                    str(value).strip().lower()
                    for value in gate_table.column(0).to_pylist()
                    if value is not None and str(value).strip()
                }
            )
            splits_found = sorted(
                {
                    str(value).strip().lower()
                    for value in gate_table.column(1).to_pylist()
                    if value is not None and str(value).strip()
                }
            )
        all_languages_present = set(LANGUAGES).issubset(languages_found)
        train_present = "train" in splits_found
        result.update(
            {
                "valid_parquet": rows > 0,
                "rows": rows,
                "columns": columns,
                "languages_found": languages_found,
                "splits_found": splits_found,
                "all_languages_present": all_languages_present,
                "train_present": train_present,
                "schema_ok": bool(
                    rows > 0
                    and "language" in normalized
                    and "split" in normalized
                    and normalized
                    & {"text", "text_norm", "transcription", "transcript"}
                    and all_languages_present
                    and train_present
                ),
            }
        )
    except Exception as exc:
        result["error"] = redact(f"{type(exc).__name__}: {exc}")
    return result

local_inventory = []
for name, path in LOCAL_CANDIDATES.items():
    exists = path.exists()
    size = path.stat().st_size if exists and path.is_file() else None
    modified_at = (
        datetime.fromtimestamp(path.stat().st_mtime, timezone.utc).isoformat()
        if exists
        else None
    )
    directory_entries = None
    if exists and path.is_dir():
        try:
            directory_entries = len(list(path.iterdir()))
        except OSError:
            directory_entries = -1
    digest = None
    if exists and path.is_file() and size is not None and size <= 256 * 2**20:
        digest = sha256_file(path)
    gate_validation = None
    if exists and path.is_file() and path.suffix.lower() == ".json":
        gate_validation = inspect_json_gate(path)
    elif exists and path.is_file() and path.suffix.lower() == ".parquet":
        gate_validation = inspect_parquet_gate(path)
    local_inventory.append(
        {
            "name": name,
            "path": str(path),
            "exists": exists,
            "kind": "directory" if exists and path.is_dir() else "file",
            "size_bytes": size,
            "modified_at": modified_at,
            "directory_entries": directory_entries,
            "sha256": digest,
            "gate_validation": gate_validation,
        }
    )

selected_records_gate = next(
    record for record in local_inventory if record["name"] == "selected_records_v4"
)
dataset_complete_gate = next(
    record
    for record in local_inventory
    if record["name"] == "balanced_dataset_complete"
)

# Fallback strict si le manifest selected_records_v4 a ete perdu mais que le
# dataset V4 materialise est encore intact : six partitions train, chacune
# avec au moins un Parquet non vide et une colonne texte. Footer uniquement.
V4_DATASET_ROOT = PROJECT_ROOT / "dataset_caches/afrivoices_balanced_v4_dataset"
V4_DATASET_VERSION_ROOT = V4_DATASET_ROOT / "version=0"
V4_LANGUAGE_DIR_CODES = {
    "sw": {"sw", "swa", "swh"},
    "kik": {"kik"},
    "kln": {"kln"},
    "luo": {"luo"},
    "som": {"som"},
    "mas": {"mas", "saq"},
}
v4_dataset_partition_snapshot: dict[str, dict[str, Any]] = {}
for language, allowed_codes in V4_LANGUAGE_DIR_CODES.items():
    matching_dirs = []
    if V4_DATASET_VERSION_ROOT.is_dir():
        for candidate in V4_DATASET_VERSION_ROOT.glob(
            "corpus=*/split=train/language=*"
        ):
            raw_name = candidate.name.split("=", 1)[-1].lower()
            code = raw_name.split("_", 1)[0]
            if code in allowed_codes:
                matching_dirs.append(candidate)
    first_parquet = None
    for directory in sorted(matching_dirs):
        try:
            first_parquet = next(directory.rglob("*.parquet"), None)
        except OSError:
            first_parquet = None
        if first_parquet is not None:
            break
    partition_record = {
        "language": language,
        "matching_directories": [str(path) for path in matching_dirs],
        "first_parquet": str(first_parquet) if first_parquet else None,
        "rows_reported_by_footer": 0,
        "columns": [],
        "schema_ok": False,
        "rows_read": 0,
        "error": None,
    }
    if first_parquet is not None:
        try:
            metadata = pq.read_metadata(first_parquet)
            columns = list(metadata.schema.to_arrow_schema().names)
            normalized_columns = {str(column).lower() for column in columns}
            partition_record.update(
                {
                    "rows_reported_by_footer": int(metadata.num_rows),
                    "columns": columns,
                    "schema_ok": bool(
                        metadata.num_rows > 0
                        and normalized_columns
                        & {"text", "text_norm", "transcription", "transcript"}
                    ),
                }
            )
        except Exception as exc:
            partition_record["error"] = redact(
                f"{type(exc).__name__}: {exc}"
            )
    v4_dataset_partition_snapshot[language] = partition_record

v4_materialized_dataset_ready = bool(
    dataset_complete_gate["exists"]
    and dataset_complete_gate["size_bytes"]
    and (dataset_complete_gate.get("gate_validation") or {}).get("valid_json")
    and not (dataset_complete_gate.get("gate_validation") or {}).get(
        "has_fail_status"
    )
    and all(
        row["schema_ok"] for row in v4_dataset_partition_snapshot.values()
    )
)
local_baseline_ready = bool(
    (selected_records_gate.get("gate_validation") or {}).get("schema_ok")
    or v4_materialized_dataset_ready
)
v5_preflight_directory = FT_ROOT / "reports/v5_preflight_20_1"
v5_preflight_candidates = [v5_preflight_directory / "LATEST.json"] + sorted(
    v5_preflight_directory.glob("v5_data_audit_*.json")
)
v5_preflight_gate_snapshot = [
    {
        "path": str(path),
        "exists": path.exists(),
        "sha256": sha256_file(path) if path.exists() and path.is_file() else None,
        "validation": inspect_json_gate(path) if path.exists() and path.is_file() else None,
    }
    for path in v5_preflight_candidates
]
v5_preflight_ready = any(
    Path(row["path"]).name.startswith("v5_data_audit_")
    and row["exists"]
    and row["validation"]["valid_json"]
    and row["validation"]["top_level_status"]
    == "PASS_READY_FOR_20_2_CTC_SEGMENTATION_PILOT"
    and not row["validation"]["has_fail_status"]
    and row["validation"]["integrity_sha_count"] > 0
    for row in v5_preflight_gate_snapshot
)


def cache_build_ids_from_report(path: Path) -> dict[str, set[str]]:
    """Extrait seulement language/cache_build_id du rapport 20.1."""
    found = {language: set() for language in LANGUAGES}
    try:
        payload = json.loads(path.read_text(encoding="utf-8"))

        def visit(value: Any) -> None:
            if isinstance(value, Mapping):
                raw_language = value.get("language")
                raw_build_id = value.get("cache_build_id")
                if raw_language is not None and raw_build_id is not None:
                    normalized = str(raw_language).strip().lower()
                    build_id = str(raw_build_id).strip()
                    for language, aliases in LANGUAGE_ALIASES.items():
                        if normalized == language or normalized in aliases:
                            if build_id and build_id.lower() not in {
                                "nan", "none", "null", "na", "n/a"
                            }:
                                found[language].add(build_id)
                for child in value.values():
                    visit(child)
            elif isinstance(value, list):
                for child in value:
                    visit(child)

        visit(payload)
    except Exception:
        pass
    return found


v5_expected_cache_build_ids = {language: set() for language in LANGUAGES}
passing_v5_reports = []
for row in v5_preflight_gate_snapshot:
    validation = row.get("validation") or {}
    if (
        row.get("exists")
        and Path(row["path"]).name.startswith("v5_data_audit_")
        and validation.get("valid_json")
        and validation.get("top_level_status")
        == "PASS_READY_FOR_20_2_CTC_SEGMENTATION_PILOT"
        and not validation.get("has_fail_status")
    ):
        passing_v5_reports.append(row)

selected_v5_preflight_report = None
if passing_v5_reports:
    selected_v5_preflight_report = max(
        passing_v5_reports,
        key=lambda row: Path(row["path"]).stat().st_mtime_ns,
    )
    extracted = cache_build_ids_from_report(
        Path(selected_v5_preflight_report["path"])
    )
    for language in LANGUAGES:
        v5_expected_cache_build_ids[language].update(extracted[language])

# Les rapports ne suffisent pas : les caches transcrits qui alimenteront 20.K2
# doivent toujours exister sur Drive. Cette verification ne lit aucune ligne.
V5_CACHE_PATTERNS = {
    "sw": "topup_sw_spont_v4_swa_speaker_audit_*",
    "kik": "topup_kik_spont_v3_textfix_longdev_*",
    "kln": "topup_kln_spont_v3_textfix_longdev_*",
    "luo": "topup_luo_spont_v3_textfix_longdev_*",
    "som": "topup_som_spont_v3_textfix_longdev_*",
    "mas": "topup_mas_spont_v3_textfix_longdev_*",
}
v5_cache_root = PROJECT_ROOT / "dataset_caches"
v5_cache_snapshot: dict[str, list[dict[str, Any]]] = {}
v5_cache_ready: dict[str, bool] = {}
for language, pattern in V5_CACHE_PATTERNS.items():
    candidates = sorted(
        path
        for path in v5_cache_root.glob(pattern)
        if path.is_dir() and ".partial" not in path.name
    )
    rows = []
    for path in candidates:
        complete_path = path / "_COMPLETE.json"
        distribution_path = path / "language_distribution_0.tsv"
        version_path = path / "version=0"
        train_directories = []
        if version_path.is_dir():
            for train_directory in version_path.glob(
                "corpus=*/split=train/language=*"
            ):
                raw_name = train_directory.name.split("=", 1)[-1].lower()
                code = raw_name.split("_", 1)[0]
                if code in V4_LANGUAGE_DIR_CODES[language]:
                    train_directories.append(train_directory)
        parquet_files = []
        for train_directory in sorted(train_directories):
            try:
                parquet_files.extend(sorted(train_directory.rglob("*.parquet")))
            except OSError:
                pass
        first_parquet = parquet_files[0] if parquet_files else None
        schema_columns = []
        rows_reported_by_first_footer = 0
        schema_ok = False
        schema_error = None
        if first_parquet is not None:
            try:
                metadata = pq.read_metadata(first_parquet)
                schema_columns = list(metadata.schema.to_arrow_schema().names)
                normalized_columns = {
                    str(column).lower() for column in schema_columns
                }
                rows_reported_by_first_footer = int(metadata.num_rows)
                schema_ok = bool(
                    metadata.num_rows > 0
                    and normalized_columns
                    & {"text", "text_norm", "transcription", "transcript"}
                )
            except Exception as exc:
                schema_error = redact(f"{type(exc).__name__}: {exc}")
        part_inventory = []
        for parquet_path in parquet_files:
            try:
                part_inventory.append(
                    {
                        "path": str(parquet_path.relative_to(path)),
                        "size_bytes": parquet_path.stat().st_size,
                    }
                )
            except OSError:
                part_inventory.append(
                    {
                        "path": str(parquet_path.relative_to(path)),
                        "size_bytes": None,
                    }
                )
        part_inventory_sha256 = sha256_bytes(
            canonical_json(part_inventory).encode("utf-8")
        )
        complete_validation = (
            inspect_json_gate(complete_path) if complete_path.is_file() else None
        )
        all_part_sizes_known_nonzero = bool(
            part_inventory
            and all(
                isinstance(item.get("size_bytes"), int)
                and item["size_bytes"] > 0
                for item in part_inventory
            )
        )
        ready = bool(
            complete_path.is_file()
            and complete_path.stat().st_size > 0
            and complete_validation
            and complete_validation.get("valid_json")
            and not complete_validation.get("has_fail_status")
            and distribution_path.is_file()
            and distribution_path.stat().st_size > 0
            and version_path.is_dir()
            and parquet_files
            and all_part_sizes_known_nonzero
            and schema_ok
        )
        rows.append(
            {
                "path": str(path),
                "ready": ready,
                "complete": str(complete_path),
                "complete_sha256": (
                    sha256_file(complete_path) if complete_path.is_file() else None
                ),
                "complete_validation": complete_validation,
                "distribution_tsv": str(distribution_path),
                "version_root": str(version_path),
                "train_directories": [str(value) for value in train_directories],
                "train_parquet_parts": len(parquet_files),
                "train_part_inventory_sha256": part_inventory_sha256,
                "all_train_part_sizes_known_nonzero": (
                    all_part_sizes_known_nonzero
                ),
                "first_parquet": str(first_parquet) if first_parquet else None,
                "first_parquet_columns": schema_columns,
                "first_parquet_rows_reported_by_footer": (
                    rows_reported_by_first_footer
                ),
                "first_parquet_schema_ok": schema_ok,
                "first_parquet_schema_error": schema_error,
                "parquet_rows_read": 0,
            }
        )
    v5_cache_snapshot[language] = rows
    ready_rows = [row for row in rows if row["ready"]]
    expected_ids = v5_expected_cache_build_ids[language]
    if len(expected_ids) == 1:
        expected_id = next(iter(expected_ids))
        selected_rows = [
            row for row in ready_rows if expected_id in Path(row["path"]).name
        ]
    elif len(expected_ids) == 0 and len(ready_rows) == 1:
        selected_rows = ready_rows
    else:
        selected_rows = []
    selected_row = selected_rows[0] if len(selected_rows) == 1 else None
    v5_cache_ready[language] = selected_row is not None
    for row in rows:
        row["selected_for_20_K2"] = bool(
            selected_row and row["path"] == selected_row["path"]
        )
        row["expected_build_ids_from_20_1"] = sorted(expected_ids)

local_languages_ready = {
    language: bool(
        local_baseline_ready
        and v5_cache_ready[language]
        and any(
            Path(row["path"]).name.startswith("v5_data_audit_")
            and row["exists"]
            and row["validation"]["valid_json"]
            and row["validation"]["top_level_status"]
            == "PASS_READY_FOR_20_2_CTC_SEGMENTATION_PILOT"
            and not row["validation"]["has_fail_status"]
            and language in row["validation"].get("languages_found", [])
            for row in v5_preflight_gate_snapshot
        )
    )
    for language in LANGUAGES
}


# -----------------------------------------------------------------------------
# 5. Authentification HF et appels metadata-only
# -----------------------------------------------------------------------------

hf_api = HfApi(token=HF_TOKEN)
hf_fs = HfFileSystem(token=HF_TOKEN)
hf_auth = {
    "token_present": bool(HF_TOKEN),
    "authenticated": False,
    "error": None,
}
if HF_TOKEN:
    try:
        hf_api.whoami(token=HF_TOKEN)
        hf_auth["authenticated"] = True
    except Exception as exc:
        hf_auth["error"] = redact(f"{type(exc).__name__}: {exc}")

HF_DATASET_SERVER = "https://datasets-server.huggingface.co"


def pinned_parquet_schema(
    repo_id: str,
    revision: str,
    filename: str,
) -> dict[str, Any]:
    """Lit uniquement le footer/schema Parquet au commit exact, jamais les lignes."""
    result = {
        "ok": False,
        "path": filename,
        "uri": None,
        "revision": revision,
        "columns": [],
        "row_groups": None,
        "rows_reported_by_footer": None,
        "footer_serialized_bytes": None,
        "rows_read": 0,
        "error": None,
    }
    if Path(filename.lower()).suffix != ".parquet":
        result["error"] = "Format non-Parquet : schema distant non sonde."
        return result
    hf_path = f"hf://datasets/{repo_id}@{revision}/{filename}"
    result["uri"] = hf_path
    try:
        if not re.fullmatch(r"[0-9a-fA-F]{40}", revision):
            raise ValueError("La revision Hub n'est pas un SHA Git complet de 40 hex.")
        resolved = hf_fs.resolve_path(hf_path)
        if str(resolved.revision).lower() != revision.lower():
            raise RuntimeError(
                "La revision interpretee par HfFileSystem differe du SHA fige."
            )
        # PyArrow ne lit ici que le footer et les metadonnees de schema. Les
        # colonnes audio/texte et leurs lignes ne sont jamais materialisees.
        with hf_fs.open(
            hf_path,
            "rb",
            block_size=256 * 1024,
            cache_type="bytes",
        ) as handle:
            metadata = pq.read_metadata(handle)
            schema = metadata.schema.to_arrow_schema()
            result.update(
                {
                    "ok": True,
                    "columns": list(schema.names),
                    "row_groups": int(metadata.num_row_groups),
                    "rows_reported_by_footer": int(metadata.num_rows),
                    "footer_serialized_bytes": getattr(
                        metadata, "serialized_size", None
                    ),
                }
            )
    except Exception as exc:
        result["error"] = redact(f"{type(exc).__name__}: {exc}")
    return result


def metadata_get(
    endpoint: str,
    repo_id: str,
    extra_params: dict[str, Any] | None = None,
    timeout: tuple[int, int] = (10, 60),
) -> dict[str, Any]:
    headers = {"Authorization": f"Bearer {HF_TOKEN}"} if HF_TOKEN else {}
    params = {"dataset": repo_id, **(extra_params or {})}
    response = None
    for attempt in range(4):
        try:
            response = requests.get(
                f"{HF_DATASET_SERVER}/{endpoint}",
                params=params,
                headers=headers,
                timeout=timeout,
            )
        except requests.RequestException as exc:
            if attempt == 3:
                return {
                    "ok": False,
                    "status_code": None,
                    "error": redact(f"{type(exc).__name__}: {exc}"),
                }
            time.sleep(min(1.5 * (2**attempt), 12.0))
            continue
        if response.status_code not in {429, 500, 502, 503, 504}:
            break
        retry_after = response.headers.get("Retry-After")
        try:
            delay = float(retry_after) if retry_after else 1.5 * (2**attempt)
        except Exception:
            delay = 1.5 * (2**attempt)
        time.sleep(min(delay, 12.0))
    if response is None:
        return {
            "ok": False,
            "status_code": None,
            "error": "Aucune reponse Dataset Server.",
        }
    if response.status_code != 200:
        return {
            "ok": False,
            "status_code": int(response.status_code),
            "error": redact(response.text[:400]),
        }
    try:
        payload = response.json()
        if isinstance(payload, dict) and payload.get("error"):
            return {
                "ok": False,
                "status_code": 200,
                "error": redact(payload.get("error")),
            }
        return {"ok": True, "status_code": 200, "payload": payload}
    except Exception as exc:
        return {
            "ok": False,
            "status_code": 200,
            "error": redact(f"JSON decode: {exc}"),
        }


def gated_value_is_true(value: Any) -> bool:
    if value is None or value is False:
        return False
    return str(value).strip().lower() not in {"", "false", "none", "0"}


def metadata_head_access(
    repo_id: str,
    revision: str,
    text_candidates: list[dict[str, Any]],
) -> dict[str, Any]:
    """Verifie un chemin via l'API Hub sans telecharger son contenu."""
    if not text_candidates:
        return {"checked": False, "ok": False, "status_code": None}
    filename = text_candidates[0]["path"]
    try:
        path_info = hf_api.get_paths_info(
            repo_id=repo_id,
            paths=[filename],
            repo_type="dataset",
            revision=revision,
            token=HF_TOKEN,
        )
        return {
            "checked": True,
            "ok": bool(path_info),
            "status_code": 200 if path_info else 404,
            "candidate_path": filename,
            "file_body_bytes_downloaded": 0,
            "method": "HfApi.get_paths_info",
        }
    except Exception as exc:
        return {
            "checked": True,
            "ok": False,
            "status_code": None,
            "candidate_path": filename,
            "file_body_bytes_downloaded": 0,
            "error": redact(f"{type(exc).__name__}: {exc}"),
        }


def card_to_dict(card_data: Any) -> dict[str, Any]:
    if card_data is None:
        return {}
    if isinstance(card_data, Mapping):
        return dict(card_data)
    if hasattr(card_data, "to_dict"):
        try:
            return dict(card_data.to_dict())
        except Exception:
            pass
    try:
        return dict(card_data)
    except Exception:
        return {}


def sibling_size(sibling: Any) -> int | None:
    value = getattr(sibling, "size", None)
    if value is not None:
        try:
            return int(value)
        except Exception:
            pass
    lfs = getattr(sibling, "lfs", None)
    if isinstance(lfs, dict) and lfs.get("size") is not None:
        try:
            return int(lfs["size"])
        except Exception:
            pass
    return None


def suffixes(path: str) -> set[str]:
    return {suffix.lower() for suffix in Path(path.lower()).suffixes}


def classify_repo_file(path: str) -> str:
    lower = path.lower()
    ext = Path(lower).suffix
    all_suffixes = suffixes(lower)
    if ext in AUDIO_EXTENSIONS:
        return "audio"
    if all_suffixes & ARCHIVE_EXTENSIONS:
        return "archive"
    if ext in TEXT_EXTENSIONS:
        if any(hint in lower for hint in TEXT_NAME_HINTS):
            return "text_manifest_likely"
        return "structured_or_text"
    return "other"


def parse_server_metadata(
    splits_result: dict[str, Any],
    parquet_result: dict[str, Any],
    validity_result: dict[str, Any],
    size_result: dict[str, Any],
    info_by_config: dict[str, dict[str, Any]],
) -> dict[str, Any]:
    def state_is_nonempty(value: Any) -> bool:
        if value is None or value is False:
            return False
        if isinstance(value, (list, tuple, set, dict)):
            return bool(value)
        return str(value).strip().lower() not in {"", "false", "none", "0", "[]", "{}"}

    def safe_url(value: Any) -> str | None:
        if not value:
            return None
        try:
            parts = urlsplit(str(value))
            return urlunsplit((parts.scheme, parts.netloc, parts.path, "", ""))
        except Exception:
            return None

    split_rows = []
    if splits_result.get("ok"):
        split_rows = splits_result.get("payload", {}).get("splits", []) or []
    configs = sorted({str(row.get("config")) for row in split_rows if row.get("config") is not None})
    splits = sorted({str(row.get("split")) for row in split_rows if row.get("split") is not None})

    parquet_rows = []
    parquet_payload: dict[str, Any] = {}
    if parquet_result.get("ok"):
        parquet_payload = parquet_result.get("payload", {})
        parquet_rows = parquet_payload.get(
            "parquet_files", parquet_payload.get("files", [])
        ) or []

    parquet_partial = state_is_nonempty(parquet_payload.get("partial"))
    parquet_pending = state_is_nonempty(parquet_payload.get("pending"))
    parquet_failed = state_is_nonempty(parquet_payload.get("failed"))
    parquet_ready = bool(
        parquet_result.get("ok")
        and parquet_rows
        and not parquet_partial
        and not parquet_pending
        and not parquet_failed
    )
    parquet_snapshot_rows = [
        {
            "config": row.get("config"),
            "split": row.get("split"),
            "filename": row.get("filename") or row.get("path"),
            "size": row.get("size"),
            "url": safe_url(row.get("url")),
        }
        for row in parquet_rows
        if isinstance(row, dict)
    ]
    parquet_snapshot_sha256 = sha256_bytes(
        canonical_json(parquet_snapshot_rows).encode("utf-8")
    )

    columns_by_config: dict[str, list[str]] = {}
    viewer_licenses: dict[str, list[str]] = {}
    for config_name, info_result in info_by_config.items():
        if not info_result.get("ok"):
            continue
        payload = info_result.get("payload", {})
        dataset_info = payload.get("dataset_info", payload)
        if not isinstance(dataset_info, dict):
            continue
        features = dataset_info.get("features", {})
        if isinstance(features, dict):
            columns_by_config[config_name] = sorted(map(str, features.keys()))
        elif isinstance(features, list):
            names = [
                item.get("name")
                for item in features
                if isinstance(item, dict) and item.get("name")
            ]
            columns_by_config[config_name] = sorted(map(str, names))
        viewer_licenses[config_name] = normalize_license(dataset_info.get("license"))

    return {
        "configs": configs,
        "splits": splits,
        "split_count": len(split_rows),
        "parquet_file_count": len(parquet_rows),
        "parquet_ready": parquet_ready,
        "parquet_partial": parquet_partial,
        "parquet_pending": parquet_pending,
        "parquet_failed": parquet_failed,
        "parquet_snapshot_sha256": parquet_snapshot_sha256,
        "dataset_server_revision_pinned": False,
        "parquet_files_preview": parquet_snapshot_rows[:50],
        "columns_by_config": columns_by_config,
        "viewer_licenses_by_config": viewer_licenses,
        "endpoint_status": {
            "splits": {k: v for k, v in splits_result.items() if k != "payload"},
            "parquet": {k: v for k, v in parquet_result.items() if k != "payload"},
            "is_valid": {k: v for k, v in validity_result.items() if k != "payload"},
            "size": {k: v for k, v in size_result.items() if k != "payload"},
            "info_by_config": {
                config: {k: v for k, v in result.items() if k != "payload"}
                for config, result in info_by_config.items()
            },
        },
    }


def inventory_hf_source(spec: dict[str, Any]) -> dict[str, Any]:
    repo_id = spec["repo_id"]
    started = time.monotonic()
    record = {
        **spec,
        "access_status": "UNKNOWN",
        "resolved_revision": None,
        "reported_licenses": [],
        "hub_reported_licenses": [],
        "viewer_reported_licenses": [],
        "viewer_license_matches_hub": None,
        "reported_languages": [],
        "language_evidence": {},
        "pinned_tree_evidence": {},
        "pinned_schema_probes": [],
        "license_status": "UNKNOWN",
        "reported_gated": None,
        "reported_private": None,
        "license_matches_expected": None,
        "content_head_access": {},
        "hub_read_access": False,
        "hub_read_access_error": None,
        "schema_evidence_ok": False,
        "train_split_ok": False,
        "expected_config_ok": False,
        "repo_file_count": 0,
        "repo_known_bytes": 0,
        "audio_file_count": 0,
        "archive_file_count": 0,
        "text_candidate_count": 0,
        "text_candidates_preview": [],
        "file_inventory_method": None,
        "files_metadata_error": None,
        "server_metadata": {},
        "error": None,
        "elapsed_seconds": None,
    }
    try:
        try:
            info = hf_api.dataset_info(
                repo_id=repo_id,
                revision="main",
                files_metadata=True,
                token=HF_TOKEN,
                timeout=120,
            )
            repo_entries = list(getattr(info, "siblings", None) or [])
            record["file_inventory_method"] = "dataset_info_files_metadata"
        except Exception as files_exc:
            # Fallback metadata-only pour les tres grands depots : l'arbre du
            # Hub est pagine et ne telecharge aucun fichier du dataset.
            record["files_metadata_error"] = redact(
                f"{type(files_exc).__name__}: {files_exc}"
            )
            info = hf_api.dataset_info(
                repo_id=repo_id,
                revision="main",
                files_metadata=False,
                token=HF_TOKEN,
                timeout=120,
            )
            repo_entries = list(
                hf_api.list_repo_tree(
                    repo_id=repo_id,
                    repo_type="dataset",
                    revision=str(getattr(info, "sha", "") or "main"),
                    recursive=True,
                    expand=False,
                    token=HF_TOKEN,
                )
            )
            record["file_inventory_method"] = "list_repo_tree_fallback"
        card = card_to_dict(getattr(info, "card_data", None))
        card_licenses = normalize_license(card.get("license"))
        tag_licenses = normalize_license(
            [
                str(tag).split(":", 1)[1]
                for tag in (getattr(info, "tags", None) or [])
                if str(tag).startswith("license:")
            ]
        )
        reported_licenses = sorted(set(card_licenses + tag_licenses))
        card_languages = string_list(card.get("language"))
        tag_languages = [
            str(tag).split(":", 1)[1]
            for tag in (getattr(info, "tags", None) or [])
            if str(tag).startswith("language:")
        ]
        reported_languages = sorted(
            {value.strip().lower() for value in card_languages + tag_languages if value}
        )
        files = []
        for sibling in repo_entries:
            if type(sibling).__name__ == "RepoFolder":
                continue
            filename = str(
                getattr(sibling, "rfilename", None)
                or getattr(sibling, "path", "")
            )
            if not filename:
                continue
            size = sibling_size(sibling)
            category = classify_repo_file(filename)
            files.append({"path": filename, "size": size, "category": category})

        text_candidates = [
            item for item in files
            if item["category"] in {"text_manifest_likely", "structured_or_text"}
        ]
        text_candidates.sort(
            key=lambda item: (
                item["category"] != "text_manifest_likely",
                item["size"] is None,
                item["size"] or 0,
                item["path"],
            )
        )

        record.update(
            {
                "access_status": "OK",
                "resolved_revision": str(getattr(info, "sha", "") or ""),
                "reported_licenses": reported_licenses,
                "hub_reported_licenses": reported_licenses,
                "reported_languages": reported_languages,
                "license_status": license_decision(
                    reported_licenses, spec.get("expected_license")
                ),
                "reported_gated": getattr(info, "gated", None),
                "reported_private": getattr(info, "private", None),
                "license_matches_expected": (
                    set(reported_licenses)
                    == set(normalize_license(spec.get("expected_license")))
                ) if reported_licenses else None,
                "repo_file_count": len(files),
                "repo_known_bytes": int(sum(item["size"] or 0 for item in files)),
                "audio_file_count": sum(item["category"] == "audio" for item in files),
                "archive_file_count": sum(item["category"] == "archive" for item in files),
                "text_candidate_count": len(text_candidates),
                "text_candidates_preview": text_candidates[:100],
            }
        )
        try:
            hf_api.auth_check(
                repo_id=repo_id,
                repo_type="dataset",
                token=HF_TOKEN,
            )
            record["hub_read_access"] = True
        except Exception as auth_exc:
            record["hub_read_access_error"] = redact(
                f"{type(auth_exc).__name__}: {auth_exc}"
            )

        # Ces endpoints retournent seulement les metadonnees de structure.
        splits_result = metadata_get("splits", repo_id)
        parquet_result = metadata_get("parquet", repo_id)
        validity_result = metadata_get("is-valid", repo_id)
        size_result = metadata_get("size", repo_id)
        split_rows = (
            splits_result.get("payload", {}).get("splits", [])
            if splits_result.get("ok")
            else []
        ) or []
        server_configs = sorted(
            {
                str(row.get("config"))
                for row in split_rows
                if isinstance(row, dict) and row.get("config") is not None
            }
        )
        requested_configs = list(spec.get("expected_configs", []))
        # Priorite aux configurations des six langues ; les grands catalogues
        # (p. ex. FLEURS) peuvent contenir des centaines de configurations.
        info_configs = list(dict.fromkeys(requested_configs + server_configs))[:30]
        info_by_config = {
            config: metadata_get("info", repo_id, {"config": config})
            for config in info_configs
        }
        record["server_metadata"] = parse_server_metadata(
            splits_result,
            parquet_result,
            validity_result,
            size_result,
            info_by_config,
        )

        observed_columns = {
            column
            for columns in record["server_metadata"]
            .get("columns_by_config", {})
            .values()
            for column in columns
        }
        expected_fields_raw = spec.get("expected_text_fields", [])
        if isinstance(expected_fields_raw, Mapping):
            expected_fields = {
                str(field).split(".")[-1]
                for fields in expected_fields_raw.values()
                for field in (fields if isinstance(fields, list) else [fields])
            }
        else:
            expected_fields = {
                str(field).split(".")[-1]
                for field in (
                    expected_fields_raw
                    if isinstance(expected_fields_raw, list)
                    else [expected_fields_raw]
                )
                if field
            }

        schema_probe_paths: list[str] = []
        config_map_for_probe = spec.get("expected_configs_by_language", {})
        for language in spec["languages"]:
            aliases = LANGUAGE_ALIASES[language]
            language_configs = {
                str(value).lower()
                for value in config_map_for_probe.get(
                    language,
                    spec.get("expected_configs", [])
                    if len(spec["languages"]) == 1
                    else [],
                )
            }
            language_candidates = []
            for item in text_candidates:
                path = item["path"]
                lower_path = path.lower()
                if Path(lower_path).suffix != ".parquet":
                    continue
                language_in_path = any(
                    re.search(
                        rf"(^|[-_/]){re.escape(alias)}($|[-_/])",
                        lower_path,
                    )
                    for alias in aliases
                )
                if not (language_in_path or len(spec["languages"]) == 1):
                    continue
                if not re.search(r"(^|[/_.-])train([/_.-]|$)", lower_path):
                    continue
                if language_configs and not any(
                    config in lower_path for config in language_configs
                ):
                    continue
                language_candidates.append(item)
            language_candidates.sort(
                key=lambda item: (item["size"] is None, item["size"] or 0, item["path"])
            )
            schema_probe_paths.extend(
                item["path"] for item in language_candidates[:2]
            )

        schema_probe_paths = list(dict.fromkeys(schema_probe_paths))
        schema_probes_by_path = {
            path: pinned_parquet_schema(
                repo_id,
                record["resolved_revision"],
                path,
            )
            for path in schema_probe_paths
        }
        record["pinned_schema_probes"] = list(schema_probes_by_path.values())
        likely_manifest_paths = [
            item["path"]
            for item in text_candidates
            if item["category"] == "text_manifest_likely"
        ]
        record["schema_evidence_ok"] = bool(
            expected_fields & observed_columns or likely_manifest_paths
        )

        observed_splits = {
            str(split).strip().lower()
            for split in record["server_metadata"].get("splits", [])
        }
        record["train_split_ok"] = bool(
            "train" in observed_splits
            or any(
                re.search(r"(^|[/_.-])train([/_.-]|$)", path.lower())
                for path in (item["path"] for item in text_candidates)
            )
        )

        expected_configs = {str(value) for value in spec.get("expected_configs", [])}
        observed_configs = set(record["server_metadata"].get("configs", []))
        record["expected_config_ok"] = bool(
            not expected_configs
            or expected_configs & observed_configs
            or any(
                any(config.lower() in item["path"].lower() for config in expected_configs)
                for item in text_candidates
            )
        )

        evidence_configs = sorted(observed_configs)
        evidence_paths = [item["path"] for item in text_candidates]
        for language in spec["languages"]:
            aliases = LANGUAGE_ALIASES[language]
            card_hits = sorted(
                {
                    value
                    for value in reported_languages
                    if value in aliases
                    or any(
                        re.search(
                            rf"(^|[-_/]){re.escape(alias)}($|[-_/])",
                            value,
                        )
                        for alias in aliases
                    )
                }
            )
            config_hits = sorted(
                {
                    config
                    for config in evidence_configs
                    if any(
                        re.search(
                            rf"(^|[-_/]){re.escape(alias)}($|[-_/])",
                            config.lower(),
                        )
                        for alias in aliases
                    )
                }
            )
            path_hits = [
                path
                for path in evidence_paths
                if any(
                    re.search(
                        rf"(^|[-_/]){re.escape(alias)}($|[-_/])",
                        path.lower(),
                    )
                    for alias in aliases
                )
            ][:10]
            record["language_evidence"][language] = {
                "ok": bool(card_hits or config_hits or path_hits),
                "card_or_tag_hits": card_hits,
                "config_hits": config_hits,
                "path_hits_preview": path_hits,
            }

            expected_files = {
                str(value).strip().lower()
                for value in spec.get("expected_text_files", [])
            }
            config_map = spec.get("expected_configs_by_language", {})
            language_configs = {
                str(value).lower()
                for value in config_map.get(
                    language,
                    spec.get("expected_configs", [])
                    if len(spec["languages"]) == 1
                    else [],
                )
            }
            correlated_paths = []
            for item in text_candidates:
                path = item["path"]
                lower_path = path.lower()
                basename = Path(lower_path).name
                language_in_path = any(
                    re.search(
                        rf"(^|[-_/]){re.escape(alias)}($|[-_/])",
                        lower_path,
                    )
                    for alias in aliases
                )
                language_ok = bool(
                    language_in_path
                    or (
                        len(spec["languages"]) == 1
                        and record["language_evidence"][language][
                            "card_or_tag_hits"
                        ]
                    )
                )
                train_ok = bool(
                    re.search(r"(^|[/_.-])train([/_.-]|$)", lower_path)
                    or (
                        expected_files
                        and basename in expected_files
                        and spec.get("expected_splits", ["train"]) == ["train"]
                    )
                )
                config_ok = bool(
                    not language_configs
                    or any(config in lower_path for config in language_configs)
                )
                schema_ok = bool(
                    expected_files and basename in expected_files
                    or (
                        schema_probes_by_path.get(path, {}).get("ok")
                        and expected_fields
                        & {
                            str(column).split(".")[-1]
                            for column in schema_probes_by_path[path].get(
                                "columns", []
                            )
                        }
                    )
                )
                if language_ok and train_ok and config_ok and schema_ok:
                    correlated_paths.append(path)
            record["pinned_tree_evidence"][language] = {
                "ok": bool(correlated_paths),
                "resolved_revision": record["resolved_revision"],
                "matching_paths_preview": correlated_paths[:10],
                "server_metadata_used": False,
            }

        viewer_licenses = sorted(
            {
                license_name
                for values in record["server_metadata"]
                .get("viewer_licenses_by_config", {})
                .values()
                for license_name in values
            }
        )
        record["viewer_reported_licenses"] = viewer_licenses
        record["viewer_license_matches_hub"] = (
            set(viewer_licenses) == set(reported_licenses)
        ) if viewer_licenses and reported_licenses else None
        # Seules la carte et les tags du commit Hub resolu gouvernent le gate.
        # La licence Dataset Server est non pincee et reste purement advisory.
        record["reported_licenses"] = reported_licenses
        record["license_status"] = license_decision(
            reported_licenses, spec.get("expected_license")
        )
        record["license_matches_expected"] = (
            set(reported_licenses)
            == set(normalize_license(spec.get("expected_license")))
        ) if reported_licenses else None

        resolved_revision = record["resolved_revision"] or "main"
        record["content_head_access"] = metadata_head_access(
            repo_id, resolved_revision, text_candidates
        )
        gated = gated_value_is_true(record["reported_gated"])
        if gated and not record["content_head_access"].get("ok"):
            record["access_status"] = "GATED_CONTENT_UNVERIFIED"
    except Exception as exc:
        name = type(exc).__name__
        message = redact(f"{name}: {exc}")
        lower = message.lower()
        if "gated" in lower or "403" in lower or "access" in lower:
            status = "GATED_NOT_ACCEPTED"
        elif "401" in lower or "unauthorized" in lower:
            status = "AUTH_FAILED"
        elif "404" in lower or "not found" in lower:
            status = "NOT_FOUND_OR_NO_ACCESS"
        else:
            status = "ERROR"
        record.update({"access_status": status, "error": message})
    record["elapsed_seconds"] = round(time.monotonic() - started, 3)
    return record


hf_inventory = []
print("\n=== INVENTAIRE HUGGING FACE, METADONNEES UNIQUEMENT ===")
for index, spec in enumerate(SOURCE_REGISTRY, 1):
    print(f"[{index:02d}/{len(SOURCE_REGISTRY):02d}] {spec['repo_id']} ...", end=" ")
    record = inventory_hf_source(spec)
    hf_inventory.append(record)
    print(
        record["access_status"],
        "| licence", record["license_status"],
        "| manifests", record["text_candidate_count"],
        "|", f"{record['elapsed_seconds']:.1f}s",
    )


# -----------------------------------------------------------------------------
# 6. Inventaire Kaggle : liste des fichiers seulement, aucun telechargement
# -----------------------------------------------------------------------------

kaggle_auth = {
    "credentials_present": KAGGLE_CREDS_PRESENT,
    "authenticated": False,
    "pages_listed": 0,
    "error": None,
}
kaggle_inventory: list[dict[str, Any]] = []

def pick_field(value: Any, *names: str) -> Any:
    for name in names:
        if isinstance(value, Mapping) and name in value:
            return value[name]
        if hasattr(value, name):
            return getattr(value, name)
    return None


if KAGGLE_CREDS_PRESENT:
    kaggle_env_names = ("KAGGLE_USERNAME", "KAGGLE_KEY", "KAGGLE_API_TOKEN")
    kaggle_env_before = {name: os.environ.get(name) for name in kaggle_env_names}
    try:
        if KAGGLE_USERNAME:
            os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
        if KAGGLE_KEY:
            os.environ["KAGGLE_KEY"] = KAGGLE_KEY
        if KAGGLE_API_TOKEN:
            os.environ["KAGGLE_API_TOKEN"] = KAGGLE_API_TOKEN

        from kaggle.api.kaggle_api_extended import KaggleApi

        kaggle_api = KaggleApi()
        kaggle_api.authenticate()
        page_token = None
        seen_tokens: set[str] = set()
        seen_names: set[str] = set()
        while True:
            response = kaggle_api.competition_list_files(
                COMPETITION,
                page_token=page_token,
                page_size=100,
            )
            kaggle_auth["pages_listed"] += 1
            if isinstance(response, (list, tuple)):
                files = list(response)
                next_token = None
            else:
                files = list(pick_field(response, "files") or [])
                next_token = pick_field(
                    response, "next_page_token", "nextPageToken"
                )
            for item in files:
                name = str(pick_field(item, "name", "ref") or "")
                if not name or name in seen_names:
                    continue
                seen_names.add(name)
                size = pick_field(item, "total_bytes", "totalBytes", "size")
                try:
                    size = int(size) if size is not None else None
                except Exception:
                    size = None
                extension = Path(name.lower()).suffix
                kaggle_inventory.append(
                    {
                        "name": name,
                        "size_bytes": size,
                        "creation_date": str(
                            pick_field(item, "creation_date", "creationDate") or ""
                        ),
                        "metadata_candidate": extension in TEXT_EXTENSIONS,
                        "archive": bool(suffixes(name) & ARCHIVE_EXTENSIONS),
                    }
                )
            if not next_token:
                break
            next_token = str(next_token)
            if next_token in seen_tokens:
                raise RuntimeError("Cycle de pagination Kaggle detecte")
            seen_tokens.add(next_token)
            page_token = next_token
        kaggle_inventory.sort(key=lambda item: item["name"])
        kaggle_auth["authenticated"] = True
    except (Exception, SystemExit) as exc:
        kaggle_auth["error"] = redact(f"{type(exc).__name__}: {exc}")
    finally:
        for name, previous in kaggle_env_before.items():
            if previous is None:
                os.environ.pop(name, None)
            else:
                os.environ[name] = previous


# -----------------------------------------------------------------------------
# 7. Couverture, controles de licence et conditions de passage a 20.K2
# -----------------------------------------------------------------------------

def usable_remote_base(record: dict[str, Any]) -> bool:
    return (
        record["access_status"] == "OK"
        and record.get("hub_read_access") is True
        and bool(record.get("resolved_revision"))
        and record.get("reported_private") is False
        and record["license_status"] == "APPROVED"
        and record.get("license_matches_expected") is not False
        and record.get("provenance_status") == "approved"
        and record["ingest_policy"] in {"extract", "mirror"}
        and record["text_candidate_count"] > 0
    )


def usable_remote_for_language(record: dict[str, Any], language: str) -> bool:
    return bool(
        usable_remote_base(record)
        and record.get("pinned_tree_evidence", {}).get(language, {}).get("ok")
    )


def usable_remote(record: dict[str, Any]) -> bool:
    return any(
        usable_remote_for_language(record, language)
        for language in record.get("languages", [])
    )


coverage_records = []
for language in LANGUAGES:
    remote = [
        record for record in hf_inventory
        if language in record["languages"]
        and usable_remote_for_language(record, language)
    ]
    manual = [
        source for source in MANUAL_EXTERNAL_SOURCES
        if language in source["languages"]
    ]
    coverage_records.append(
        {
            "language": language,
            "local_v4_ready": local_baseline_ready,
            "local_v5_text_ready": local_languages_ready[language],
            "remote_approved_sources": len(remote),
            "remote_source_ids": [record["source_id"] for record in remote],
            "remote_source_families": sorted(
                {record["family"] for record in remote}
            ),
            "manual_optional_sources": len(manual),
            "manual_source_ids": [source["source_id"] for source in manual],
            "ready_for_text_extraction": bool(
                local_languages_ready[language] or remote
            ),
        }
    )

blockers: list[dict[str, Any]] = []
warnings: list[dict[str, Any]] = []

if not HF_TOKEN:
    blockers.append(
        {
            "code": "HF_TOKEN_MISSING",
            "detail": "Ajouter un nouveau HF_TOKEN read-only dans Colab Secrets.",
        }
    )
elif not hf_auth["authenticated"]:
    blockers.append(
        {
            "code": "HF_AUTH_FAILED",
            "detail": hf_auth["error"] or "Authentification Hugging Face impossible.",
        }
    )

anv_main = next(
    record for record in hf_inventory if record["source_id"] == "anv_ke_catalog"
)
anv_mirrors_ok = {
    language: any(
        record["family"] == "anv_ke"
        and record["ingest_policy"] == "mirror"
        and language in record["languages"]
        and usable_remote_for_language(record, language)
        for record in hf_inventory
    )
    for language in ("kik", "kln", "luo", "som", "mas")
}
for language, mirror_ok in anv_mirrors_ok.items():
    if not (
        local_languages_ready[language]
        or usable_remote_for_language(anv_main, language)
        or mirror_ok
    ):
        blockers.append(
            {
                "code": "ANV_TEXT_ACCESS_MISSING",
                "language": language,
                "detail": (
                    "Accepter les conditions du depot MCAA1-MSU/anv_data_ke "
                    "ou de son miroir Anv-ke correspondant."
                ),
                "url": "https://huggingface.co/datasets/MCAA1-MSU/anv_data_ke",
            }
        )

sw_main = next(
    record for record in hf_inventory if record["source_id"] == "afrivoice_swahili"
)
if not usable_remote_for_language(sw_main, "sw"):
    warnings.append(
        {
            "code": "AFRIVOICE_SW_FULL_TEXT_UNAVAILABLE",
            "detail": (
                "Le V4 local reste disponible, mais accepter l'acces AfriVoice "
                "Swahili permet d'utiliser tout le train transcrit en 20.K2."
            ),
            "url": "https://huggingface.co/datasets/DigitalUmuganda/Afrivoice_Swahili",
        }
    )

for record in hf_inventory:
    if record["access_status"] != "OK":
        warnings.append(
            {
                "code": "HF_SOURCE_UNAVAILABLE",
                "source_id": record["source_id"],
                "repo_id": record["repo_id"],
                "access_status": record["access_status"],
                "detail": record.get("error"),
            }
        )
    elif record.get("hub_read_access") is not True:
        warnings.append(
            {
                "code": "HF_CONTENT_ACCESS_UNVERIFIED",
                "source_id": record["source_id"],
                "repo_id": record["repo_id"],
                "detail": record.get("hub_read_access_error"),
            }
        )
    elif record.get("reported_private") is not False:
        warnings.append(
            {
                "code": "PRIVATE_SOURCE_NOT_ALLOWED",
                "source_id": record["source_id"],
                "repo_id": record["repo_id"],
                "detail": "Les donnees externes doivent etre publiquement accessibles.",
            }
        )
    elif record["license_status"] != "APPROVED":
        warnings.append(
            {
                "code": "LICENSE_NOT_AUTO_APPROVED",
                "source_id": record["source_id"],
                "repo_id": record["repo_id"],
                "reported": record["reported_licenses"],
                "expected": record.get("expected_license"),
                "decision": record["license_status"],
                "detail": "Ne pas ingerer cette source automatiquement en 20.K2.",
            }
        )

    failed_evidence = [
        name
        for name in ("schema_evidence_ok", "train_split_ok", "expected_config_ok")
        if record.get(name) is not True
    ]
    if failed_evidence:
        warnings.append(
            {
                "code": "TEXT_EXTRACTION_EVIDENCE_INCOMPLETE",
                "source_id": record["source_id"],
                "repo_id": record["repo_id"],
                "failed_checks": failed_evidence,
            }
        )
    missing_language_evidence = [
        language
        for language in record["languages"]
        if not record.get("language_evidence", {}).get(language, {}).get("ok")
    ]
    if missing_language_evidence:
        warnings.append(
            {
                "code": "LANGUAGE_EVIDENCE_INCOMPLETE",
                "source_id": record["source_id"],
                "repo_id": record["repo_id"],
                "languages": missing_language_evidence,
            }
        )
    elif record.get("license_matches_expected") is False:
        warnings.append(
            {
                "code": "LICENSE_DECLARATION_MISMATCH",
                "source_id": record["source_id"],
                "repo_id": record["repo_id"],
                "reported": record["reported_licenses"],
                "expected": record.get("expected_license"),
                "detail": "Revue juridique/provenance requise avant 20.K2.",
            }
        )

    if record.get("provenance_status") != "approved":
        warnings.append(
            {
                "code": "PROVENANCE_REVIEW_REQUIRED",
                "source_id": record["source_id"],
                "repo_id": record["repo_id"],
                "detail": "Source inventoriee mais desactivee par defaut.",
            }
        )
    if record.get("viewer_license_matches_hub") is False:
        warnings.append(
            {
                "code": "VIEWER_LICENSE_DIFFERS_FROM_PINNED_HUB",
                "source_id": record["source_id"],
                "repo_id": record["repo_id"],
                "hub": record.get("hub_reported_licenses"),
                "viewer_advisory": record.get("viewer_reported_licenses"),
                "detail": "Le Dataset Server n'est pas utilise pour autoriser la source.",
            }
        )

for row in coverage_records:
    if not row["ready_for_text_extraction"]:
        blockers.append(
            {
                "code": "LANGUAGE_WITHOUT_TEXT_SOURCE",
                "language": row["language"],
            }
        )

if not local_baseline_ready:
    blockers.append(
        {
            "code": "LOCAL_V4_BASELINE_MISSING",
            "detail": "Restaurer selected_records_v4 ou le dataset V4 sur Drive.",
        }
    )

if not v5_preflight_ready:
    blockers.append(
        {
            "code": "V5_PREFLIGHT_20_1_MISSING",
            "detail": "Le rapport fige de 20.1 doit exister sur Drive avant 20.K2.",
        }
    )

if not kaggle_auth["authenticated"]:
    warnings.append(
        {
            "code": "KAGGLE_LISTING_UNAVAILABLE",
            "detail": kaggle_auth["error"] or "Ajouter KAGGLE_USERNAME/KAGGLE_KEY.",
        }
    )

for source in MANUAL_EXTERNAL_SOURCES:
    warnings.append(
        {
            "code": "MANUAL_TEXT_EXPORT_OPTIONAL",
            "source_id": source["source_id"],
            "languages": source["languages"],
            "url": source["url"],
            "detail": source["notes"],
        }
    )

status = "PASS_READY_FOR_20_K2" if not blockers else "REVIEW"
parquet_footer_probes_ok = sum(
    1
    for record in hf_inventory
    for probe in record.get("pinned_schema_probes", [])
    if probe.get("ok")
)
parquet_footer_serialized_bytes_reported = sum(
    int(probe.get("footer_serialized_bytes") or 0)
    for record in hf_inventory
    for probe in record.get("pinned_schema_probes", [])
    if probe.get("ok")
)


# -----------------------------------------------------------------------------
# 8. Contrat et rapports immuables sur Drive
# -----------------------------------------------------------------------------

hf_observation = [
    {
        "source_id": record["source_id"],
        "access_status": record["access_status"],
        "hub_read_access": record["hub_read_access"],
        "resolved_revision": record["resolved_revision"],
        "reported_licenses": record["reported_licenses"],
        "license_status": record["license_status"],
        "reported_private": record["reported_private"],
        "reported_gated": record["reported_gated"],
        "language_evidence": record["language_evidence"],
        "pinned_tree_evidence": record["pinned_tree_evidence"],
        "schema_evidence_ok": record["schema_evidence_ok"],
        "train_split_ok": record["train_split_ok"],
        "expected_config_ok": record["expected_config_ok"],
        "repo_file_count": record["repo_file_count"],
        "repo_known_bytes": record["repo_known_bytes"],
        "parquet_snapshot_sha256": record.get("server_metadata", {}).get(
            "parquet_snapshot_sha256"
        ),
        "parquet_ready": record.get("server_metadata", {}).get("parquet_ready"),
    }
    for record in hf_inventory
]
hf_observation_sha256 = sha256_bytes(
    canonical_json(hf_observation).encode("utf-8")
)
kaggle_inventory_sha256 = sha256_bytes(
    canonical_json(kaggle_inventory).encode("utf-8")
)

inventory_contract = {
    "schema": 2,
    "cell": "20.K1",
    "competition": COMPETITION,
    "languages": LANGUAGES,
    "source_registry_sha256": sha256_bytes(
        canonical_json(SOURCE_REGISTRY).encode("utf-8")
    ),
    "hf_observation_sha256": hf_observation_sha256,
    "kaggle_inventory_sha256": kaggle_inventory_sha256,
    "dataset_server_snapshot_sha256": {
        record["source_id"]: record.get("server_metadata", {}).get(
            "parquet_snapshot_sha256"
        )
        for record in hf_inventory
    },
    "dataset_server_revision_pinned": False,
    "status": status,
    "blockers_sha256": sha256_bytes(
        canonical_json(blockers).encode("utf-8")
    ),
    "coverage_sha256": sha256_bytes(
        canonical_json(coverage_records).encode("utf-8")
    ),
    "local_gate_snapshot": {
        "local_baseline_ready": local_baseline_ready,
        "v4_materialized_dataset_ready": v4_materialized_dataset_ready,
        "v4_dataset_partition_snapshot": v4_dataset_partition_snapshot,
        "local_languages_ready": local_languages_ready,
        "v5_cache_ready": v5_cache_ready,
        "v5_cache_snapshot": v5_cache_snapshot,
        "v5_preflight_ready": v5_preflight_ready,
        "selected_v5_preflight_report": selected_v5_preflight_report,
        "v5_preflight": v5_preflight_gate_snapshot,
        "artifacts": [
            {
                "name": record["name"],
                "exists": record["exists"],
                "size_bytes": record["size_bytes"],
                "modified_at": record["modified_at"],
                "sha256": record["sha256"],
                "gate_validation": record["gate_validation"],
            }
            for record in local_inventory
            if record["name"] in {
                "selected_records_v4",
                "balanced_dataset_complete",
                "v5_preflight_latest",
            }
        ],
    },
    "resolved_revisions": {
        record["source_id"]: record["resolved_revision"]
        for record in hf_inventory
    },
    "local_artifact_sha256": {
        record["name"]: record["sha256"]
        for record in local_inventory
        if record["sha256"]
    },
    "network_policy": {
        "hf_repo_metadata_only": True,
        "hf_dataset_server_metadata_only": True,
        "kaggle_list_files_only": True,
        "dataset_row_payload_downloaded": False,
        "audio_downloaded": False,
        "row_text_sampled": False,
        "dataset_rows_read": 0,
        "dataset_row_payload_downloaded_bytes": 0,
        "parquet_schema_footer_probes_ok": parquet_footer_probes_ok,
        "parquet_footer_serialized_bytes_reported": (
            parquet_footer_serialized_bytes_reported
        ),
        "parquet_footer_transport_bytes_metered": False,
        "audio_downloaded_bytes": 0,
        "dataset_file_body_downloaded_bytes": 0,
    },
    "future_dedup_policy": {
        "one_source_per_family_before_concat": True,
        "mirrors_mutually_exclusive": True,
        "train_split_only": True,
        "cross_source_exact_and_near_dedup_required_in_20_K2": True,
    },
}
contract_sha256 = sha256_bytes(canonical_json(inventory_contract).encode("utf-8"))
run_id = contract_sha256[:16]
FINAL_RUN_DIR = REPORT_ROOT / run_id
staging_parent = Path("/content") if Path("/content").exists() else Path(tempfile.gettempdir())
RUN_DIR = Path(
    tempfile.mkdtemp(prefix=f"kenlm_20_K1_{run_id}_", dir=staging_parent)
)

report = {
    "schema": 2,
    "cell": "20.K1",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "status": status,
    "run_id": run_id,
    "contract_sha256": contract_sha256,
    "contract": inventory_contract,
    "security": {
        "secrets_written_to_report": False,
        "secret_values_printed": False,
        "hf_token_present": bool(HF_TOKEN),
        "hf_authenticated": hf_auth["authenticated"],
        "kaggle_credentials_present": KAGGLE_CREDS_PRESENT,
        "kaggle_authenticated": kaggle_auth["authenticated"],
        "recommendation": (
            "Utiliser uniquement de nouveaux secrets read-only dans Colab Secrets ; "
            "ne jamais les enregistrer sur Drive."
        ),
    },
    "local_inventory": local_inventory,
    "v5_cache_snapshot": v5_cache_snapshot,
    "hf_inventory": hf_inventory,
    "manual_external_sources": MANUAL_EXTERNAL_SOURCES,
    "kaggle_inventory": kaggle_inventory,
    "coverage": coverage_records,
    "blockers": blockers,
    "warnings": warnings,
    "next_step": (
        "20.K2_stream_normalize_deduplicate_text"
        if status == "PASS_READY_FOR_20_K2"
        else "Corriger les blockers puis relancer 20.K1"
    ),
}

# Garde-fou final : aucune valeur de secret ne peut apparaitre dans le rapport.
serialized_report = canonical_json(report)
for secret in SECRET_VALUES:
    if secret and len(secret) >= 4:
        if secret in serialized_report:
            raise RuntimeError("FAIL_SECRET_LEAK: valeur secrete dans le rapport.")
for sensitive_pattern in (
    r"\bhf_[A-Za-z0-9]{20,}\b",
    r"(?i)\bBearer\s+[A-Za-z0-9._-]{10,}",
    r"(?i)https?://[^\s\"']+[?&](?:token|key|authorization)=[^&\s\"']+",
):
    if re.search(sensitive_pattern, serialized_report):
        raise RuntimeError("FAIL_SECRET_LEAK: motif sensible dans le rapport.")

report_path = RUN_DIR / "kenlm_text_source_inventory_20_K1.json"
hf_csv_path = RUN_DIR / "hf_source_inventory_20_K1.csv"
coverage_csv_path = RUN_DIR / "language_coverage_20_K1.csv"
kaggle_csv_path = RUN_DIR / "kaggle_file_inventory_20_K1.csv"

atomic_json(report, report_path)
atomic_csv(
    [
        {
            "source_id": row["source_id"],
            "repo_id": row["repo_id"],
            "languages": ",".join(row["languages"]),
            "priority": row["priority"],
            "core": row["core"],
            "kind": row["kind"],
            "ingest_policy": row["ingest_policy"],
            "provenance_status": row["provenance_status"],
            "access_status": row["access_status"],
            "revision": row["resolved_revision"],
            "reported_licenses": ",".join(row["reported_licenses"]),
            "license_status": row["license_status"],
            "license_matches_expected": row["license_matches_expected"],
            "reported_gated": row["reported_gated"],
            "content_head_ok": row.get("content_head_access", {}).get("ok"),
            "repo_file_count": row["repo_file_count"],
            "repo_known_gib": round(row["repo_known_bytes"] / 2**30, 4),
            "text_candidate_count": row["text_candidate_count"],
            "configs": ",".join(row.get("server_metadata", {}).get("configs", [])),
            "splits": ",".join(row.get("server_metadata", {}).get("splits", [])),
            "parquet_file_count": row.get("server_metadata", {}).get("parquet_file_count", 0),
            "error": row["error"],
        }
        for row in hf_inventory
    ],
    hf_csv_path,
)
atomic_csv(coverage_records, coverage_csv_path)
atomic_csv(kaggle_inventory, kaggle_csv_path)

report_sha256 = sha256_file(report_path)
complete_path = RUN_DIR / "_COMPLETE.json"
complete = {
    "schema": 2,
    "cell": "20.K1",
    "status": status,
    "run_id": run_id,
    "contract_sha256": contract_sha256,
    "artifacts": {
        str(path.name): {
            "bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        }
        for path in (
            report_path,
            hf_csv_path,
            coverage_csv_path,
            kaggle_csv_path,
        )
    },
    "metadata_only": True,
    "audio_downloaded_bytes": 0,
    "dataset_rows_read": 0,
    "dataset_row_payload_downloaded_bytes": 0,
    "parquet_schema_footer_probes_ok": parquet_footer_probes_ok,
}
atomic_json(complete, complete_path)

# Scanner tous les artefacts de staging, pas seulement le JSON principal.
artifact_paths = [
    report_path,
    hf_csv_path,
    coverage_csv_path,
    kaggle_csv_path,
    complete_path,
]
for artifact_path in artifact_paths:
    artifact_text = artifact_path.read_text(encoding="utf-8", errors="replace")
    for secret in SECRET_VALUES:
        if secret and len(secret) >= 4 and secret in artifact_text:
            shutil.rmtree(RUN_DIR, ignore_errors=True)
            raise RuntimeError(
                f"FAIL_SECRET_LEAK: valeur secrete dans {artifact_path.name}."
            )
    for pattern in (
        r"\bhf_[A-Za-z0-9]{20,}\b",
        r"(?i)\bBearer\s+[A-Za-z0-9._-]{10,}",
        r"(?i)https?://[^\s\"']+[?&](?:token|key|authorization)=[^&\s\"']+",
    ):
        if re.search(pattern, artifact_text):
            shutil.rmtree(RUN_DIR, ignore_errors=True)
            raise RuntimeError(
                f"FAIL_SECRET_LEAK: motif sensible dans {artifact_path.name}."
            )

# Publication atomique et immuable. Un contrat deja complet est reutilise ;
# un ancien dossier incomplet est mis en quarantaine, jamais confondu avec PASS.
reuse_existing = False
if (FINAL_RUN_DIR / "_COMPLETE.json").exists():
    try:
        existing_complete = json.loads(
            (FINAL_RUN_DIR / "_COMPLETE.json").read_text(encoding="utf-8")
        )
        expected_artifact_names = {
            "kenlm_text_source_inventory_20_K1.json",
            "hf_source_inventory_20_K1.csv",
            "language_coverage_20_K1.csv",
            "kaggle_file_inventory_20_K1.csv",
        }
        existing_artifacts = existing_complete.get("artifacts", {})
        existing_report = json.loads(
            (FINAL_RUN_DIR / "kenlm_text_source_inventory_20_K1.json").read_text(
                encoding="utf-8"
            )
        )
        reuse_existing = (
            existing_complete.get("contract_sha256") == contract_sha256
            and existing_complete.get("run_id") == run_id
            and existing_complete.get("status") == status
            and existing_report.get("status") == status
            and existing_report.get("contract_sha256") == contract_sha256
            and set(existing_artifacts) == expected_artifact_names
            and all(
                (FINAL_RUN_DIR / name).exists()
                and (FINAL_RUN_DIR / name).stat().st_size == meta.get("bytes")
                and sha256_file(FINAL_RUN_DIR / name) == meta.get("sha256")
                for name, meta in existing_artifacts.items()
            )
        )
    except Exception:
        reuse_existing = False

if reuse_existing:
    shutil.rmtree(RUN_DIR, ignore_errors=True)
else:
    if FINAL_RUN_DIR.exists():
        quarantine = REPORT_ROOT / (
            f"{run_id}.incomplete-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
        )
        os.replace(FINAL_RUN_DIR, quarantine)
    shutil.move(str(RUN_DIR), str(FINAL_RUN_DIR))

report_path = FINAL_RUN_DIR / report_path.name
hf_csv_path = FINAL_RUN_DIR / hf_csv_path.name
coverage_csv_path = FINAL_RUN_DIR / coverage_csv_path.name
kaggle_csv_path = FINAL_RUN_DIR / kaggle_csv_path.name
complete_path = FINAL_RUN_DIR / complete_path.name
report_sha256 = sha256_file(report_path)

latest = {
    "schema": 2,
    "cell": "20.K1",
    "status": status,
    "run_id": run_id,
    "contract_sha256": contract_sha256,
    "report": str(report_path),
    "report_sha256": report_sha256,
    "hf_inventory_csv": str(hf_csv_path),
    "coverage_csv": str(coverage_csv_path),
    "kaggle_inventory_csv": str(kaggle_csv_path),
    "complete": str(complete_path),
    "complete_sha256": sha256_file(complete_path),
    "audio_downloaded": False,
    "dataset_row_payload_downloaded": False,
    "audio_downloaded_bytes": 0,
    "dataset_rows_read": 0,
    "dataset_row_payload_downloaded_bytes": 0,
    "parquet_schema_footer_probes_ok": parquet_footer_probes_ok,
}
atomic_json(latest, REPORT_ROOT / "LATEST_ATTEMPT.json")
atomic_json(latest, REPORT_ROOT / "LATEST.json")
if status == "PASS_READY_FOR_20_K2":
    atomic_json(latest, REPORT_ROOT / "LATEST_PASS.json")

# Retirer les secrets et clients du namespace du notebook apres les requetes.
HF_TOKEN = None
KAGGLE_USERNAME = None
KAGGLE_KEY = None
KAGGLE_API_TOKEN = None
SECRET_VALUES.clear()
for _client_name in ("hf_api", "hf_fs", "kaggle_api"):
    globals().pop(_client_name, None)


# -----------------------------------------------------------------------------
# 9. Sortie lisible, sans texte de corpus et sans secret
# -----------------------------------------------------------------------------

summary_rows = []
for row in hf_inventory:
    summary_rows.append(
        {
            "source": row["source_id"],
            "languages": ",".join(row["languages"]),
            "access": row["access_status"],
            "license": ",".join(row["reported_licenses"]) or row.get("expected_license"),
            "decision": row["license_status"],
            "provenance": row["provenance_status"],
            "manifests": row["text_candidate_count"],
            "parquets": row.get("server_metadata", {}).get("parquet_file_count", 0),
            "policy": row["ingest_policy"],
        }
    )

print("\n=== SOURCES KENLM V6 ===")
try:
    from IPython.display import display

    display(pd.DataFrame(summary_rows))
    print("\n=== COUVERTURE PAR LANGUE ===")
    display(pd.DataFrame(coverage_records))
except Exception:
    print(pd.DataFrame(summary_rows).to_string(index=False))
    print(pd.DataFrame(coverage_records).to_string(index=False))

print("\nSTATUT 20.K1 :", status)
print("Audio telecharge : 0")
print("Lignes/payload de colonnes lus : 0")
print("Footers Parquet sondes au SHA exact :", parquet_footer_probes_ok)
print("Rapport :", report_path)
print("SHA256  :", report_sha256)

if blockers:
    print("\nBLOQUANTS :")
    for item in blockers:
        print(" -", item)
if warnings:
    print("\nWARNINGS :", len(warnings), "(details dans le rapport)")

if status == "PASS_READY_FOR_20_K2":
    print("\n✅ 20.K1 termine. Etape suivante : 20.K2 text-only streaming + normalisation + dedup.")
else:
    print("\nCorriger les bloquants puis relancer exactement cette cellule.")
